# Component 2: INT8 quantisation (Colab)

Derives the INT8 deployment version of the frozen model, following `docs/QUANTIZATION.md` (`int8-v1`).

- **Float32 preprocessing + INT8 neural inference.** The float contract in `models/` is not changed; the INT8 parameters are a deployment derivative.
- **TRAINING rows only.** OPERATION rows are dropped straight after loading and are never used.
- Nothing is trained or refitted. Thresholds, features and labels are unchanged.
- The host C tests are compiled with gcc in this session. **No ESP32 measurement is made**, so no memory, latency or energy result is claimed.

All code comes from `src/tinyml/` (`quantize.py`, `device_reference.py`, `golden.py`, `codegen.py`) and `scripts/07_quantize_model.py`. This notebook only calls it.

Run the cells one at a time with **Shift + Enter** (not "Run all").

## Step 1: Get the latest repository code and install dependencies

In [ ]:
# STEP 1: Get the latest component2-tinyml code and install dependencies
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True, check=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)
print(subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## Step 2: Import the modules and verify the frozen model (no data is read)

In [ ]:
# STEP 2: Import modules and verify the committed model artifacts
import importlib.util
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.train import check_dataset_sha256
from tinyml.holdout import verify_frozen_model
from tinyml.quantize import QUANT_VERSION

MODEL_DIR = f"{COMPONENT_DIR}/models"
contract = verify_frozen_model(MODEL_DIR)          # pinned SHA-256 values; stops on any difference
print("Quantisation:", QUANT_VERSION)
print("Model       :", contract["model_version"], "| status", contract["status"],
      "| weights", contract["weights_sha256"][:16], "...")
print("Frozen model verified.")

spec = importlib.util.spec_from_file_location("quant", f"{COMPONENT_DIR}/scripts/07_quantize_model.py")
quant = importlib.util.module_from_spec(spec); spec.loader.exec_module(quant)

## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. Only the TRAINING rows are used.

In [ ]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

## Step 4: Check the dataset fingerprint (stops if it is not the validated file)

In [ ]:
# STEP 4: SHA-256 check
sha = check_dataset_sha256(file_sha256(DATA_PATH))
print("SHA-256 OK:", sha)

## Step 5: Quantise (TRAINING rows only)
Calibrates on status-OK TRAINING rows, compares INT8 with the float reference, builds the golden vectors, generates the C headers and runs the host C tests. Writes everything to `/content/quant_out/` and refuses to overwrite earlier output.

In [ ]:
# STEP 5: INT8 quantisation (no OPERATION data)
SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
OUT_ROOT = "/content/quant_out"
report, q, g = quant.run(DATA_PATH, SPLITS_PATH, MODEL_DIR, OUT_ROOT, code_commit=COMMIT)
a = report["acceptance"]
for k, v in a["checks"].items():
    print(f"{k:28s} {'PASS' if v else 'FAIL'}")
print("INT8 accepted:", a["int8_accepted"], "|", a["deployment_candidate"])
print("Host C tests :", report["host_c_tests"]["output"][-120:])

## Step 6: Show the report

In [ ]:
# STEP 6: Show the Markdown report
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT_ROOT}/results/quantization_report.md", encoding="utf-8").read()))

## Step 7: Download the results
Downloads `quant_out.zip` (INT8 parameters, manifest, generated C headers, golden vectors, report). **Send it back for review.** Nothing is committed until it is reviewed and approved.

In [ ]:
# STEP 7: Zip and download
import shutil
zip_path = shutil.make_archive("/content/quant_out", "zip", OUT_ROOT)
files.download(zip_path)
print("Downloaded", zip_path)